# Stage 08 — C1: ICDR two-route head vs CORN refit on the frozen representation

**Question.** Does the sequential CORN output structure itself contribute to PDR under-triage? CORN's
task *k* is trained on grades ≥ *k*, so every grade-4 label supervises the NPDR tasks 0–2 and a
grade-4 prediction is reachable only through them. ICDR defines PDR disjunctively (NV or
vitreous/preretinal haemorrhage), independent of NPDR lesion burden.

**Controlled comparison on the identical frozen Stage-7 representation E** (NO_RACAF BEST backbones,
seeds 42 / 123 / 2026; Stages 1–7 frozen; nothing retrained):

| head | architecture | parameters | role |
|---|---|---|---|
| **H0** | original jointly trained CORN head (saved outputs) | 1,028 | continuity |
| **H1** | CORN refit, Dense(256→4), grades 0–4 | 1,028 | control |
| **H2** | PDR route Dense(256→1) → q = P(4); NPDR CORN route Dense(256→3) over grades 0–3 only; P(k) = (1 − q)·P_NPDR(k); 4 if q > 0.5 else CORN decode over 0–3 | 1,028 | treatment |

Fitting (identical for H1 and H2, pre-registered): one L2-regularised class-weighted logistic fitter
for every task, L2 = 1e-4, square-root inverse-frequency class weights, L-BFGS from zero
(deterministic), training-only standardisation, q threshold 0.5. Fitted on the cached training split
only (the split minus the empty-field-of-view images pinned in the six-run manifest -- the population
the backbones were trained on; the NPDR route uses its grade 0–3 images); validation (730) is scored
once.

**Primary endpoint.** Per seed, Δ = AUROC(H2) − AUROC(H1) for grade 4 vs grades 0–2, each head
scored by its own P(grade ≥ 3). **SUPPORTIVE** if mean Δ ≥ +0.03, Δ > 0 in 3/3 seeds and the
guardrails hold; **NOT_SUPPORTIVE** if mean Δ < +0.01 or Δ ≤ 0 in ≥ 2 seeds; otherwise
**TRADE_OFF** / **INCONCLUSIVE**. Paired stratified bootstrap: 2,000 resamples, seed 20260927.

C1 is **not** a new ordinal model family (a split-first / hurdle structure is established). **DDR is
not used** (reserved for later external evaluation); IDRiD is not used.

**How to use.** Select a T4 GPU runtime and *Run all*. The notebook freezes `PREREGISTRATION.json`
before anything is extracted or fitted, re-runs the C1 test suite on this runtime, then extracts E seed
by seed and fits the heads. After a disconnect, *Run all* again: every seed whose E was already
extracted and verified is reused, and the analysis resumes. The experiment is single-use — once
`results.json` exists, *Run all* only reprints the stored report. Nothing is edited by hand between
sessions. Implementation: `icdr_two_route_head.py`, `icdr_two_route_experiment.py`.

In [ ]:
# ==== [1] ENVIRONMENT / IMPORTS ====
import os
import posixpath
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import setup as colab_setup
setup_info = colab_setup.setup()

import colab_config
import verify_environment
env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR, drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"), require_gpu=True)

import datetime
import json
import time

import tensorflow as tf

import config
import icdr_two_route_experiment as exp
import icdr_two_route_head as th
import improved_training_data as itd
import multiseed_runs as msr

GPUS = tf.config.list_physical_devices("GPU")
if not GPUS:
    raise RuntimeError("No GPU: select a T4 runtime (mixed_float16 parity with the saved "
                       "predictions requires it).")
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
print(f"Ready. C1 {exp.EXPERIMENT_VERSION} / head {th.HEAD_VERSION}; multiseed_runs protocol "
      f"{msr.PROTOCOL_VERSION}; repo {REPO_COMMIT}; GPUs {GPUS}")

In [ ]:
# ==== [2] CONFIGURATION AND FROZEN PRE-REGISTRATION ====
# Written BEFORE any extraction or fit. On every later session it is re-derived from the code and
# compared; any difference (fitter, L2, weights, threshold, rule, endpoint, seeds or the C1 source
# files) stops the notebook.
EXPERIMENTS_ROOT = colab_config.DRIVE.experiments_root
SIX_RUN_ROOT = posixpath.join(EXPERIMENTS_ROOT, exp.SIX_RUN_SUBDIR)
SIX_RUN_DIR = msr.experiment_root(SIX_RUN_ROOT, exp.SIX_RUN_ID)
GROUP_DIR = posixpath.join(EXPERIMENTS_ROOT, exp.OUTPUT_GROUP)

# One timestamped experiment directory, created on the first session and resumed afterwards.
EXPERIMENT_DIR, COMPLETED = exp.resolve_experiment_dir(GROUP_DIR)
for _protected in (SIX_RUN_DIR, posixpath.join(EXPERIMENTS_ROOT, "Grade4AuxLoss"),
                   posixpath.join(EXPERIMENTS_ROOT, "Grade3vs4_Phase0"),
                   posixpath.join(EXPERIMENTS_ROOT, "Task3WeightingPlacebo"),
                   posixpath.join(EXPERIMENTS_ROOT, "ImprovedTrainingC1Control"),
                   posixpath.join(EXPERIMENTS_ROOT, "ImprovedTrainingC3Kappa"),
                   colab_config.DRIVE.experiment_dir("FinalClassification")):
    assert posixpath.commonpath([posixpath.normpath(EXPERIMENT_DIR),
                                 posixpath.normpath(_protected)]) != posixpath.normpath(_protected), (
        f"{EXPERIMENT_DIR} resolves under protected path {_protected} -- refusing.")

PREREGISTRATION, _how = exp.freeze_or_verify_preregistration(EXPERIMENT_DIR, REPO_DIR, REPO_COMMIT)
print(f"Experiment directory: {EXPERIMENT_DIR}")
print(f"Pre-registration {_how} (first written at commit {PREREGISTRATION['repo_commit']}, "
      f"{PREREGISTRATION['written']}); this session's commit {REPO_COMMIT}")
print(f"Seeds {exp.BACKBONE_SEEDS}; L2 {th.L2}; class weights {[round(w, 4) for w in th.CLASS_WEIGHTS]}; "
      f"q threshold {th.PDR_DECISION_THRESHOLD}; bootstrap {exp.N_BOOTSTRAP} x seed {exp.BOOTSTRAP_SEED}")
print(f"Primary endpoint: {PREREGISTRATION['primary_endpoint']}")
for _k, _v in PREREGISTRATION["decision_rule"].items():
    print(f"  {_k}: {_v}")
print("Out of scope:", PREREGISTRATION["out_of_scope"])
if COMPLETED:
    print("\nThis experiment is COMPLETED (single-use). Cells [3]-[7] are skipped; [8] reprints the "
          "stored report.")

In [ ]:
# ==== [3] PRE-RUN VERIFICATION (every session, before any extraction or fit) ====
# The C1 test suite is re-run on THIS runtime's library versions, so a version difference can never
# silently change the heads, the fitter, the decode, the metrics or the decision rule.
if not COMPLETED:
    PARAMETER_PARITY = th.parameter_parity_report()      # raises unless H1 == H2 == 1,028
    print("Parameter parity:", json.dumps(PARAMETER_PARITY))
    # Run BY FILE PATH with the repo root on PYTHONPATH: Colab's Python ships an unrelated
    # site-packages package named `tests`, which would shadow the repo's tests/ directory if the
    # suite were imported as `tests.test_icdr_two_route_head`.
    _env = dict(os.environ, PYTHONPATH=os.pathsep.join(
        [REPO_DIR, os.path.join(REPO_DIR, "colab", "common"), os.environ.get("PYTHONPATH", "")]))
    _t0 = time.time()
    _tests = subprocess.run([sys.executable, os.path.join(REPO_DIR, "tests",
                                                          "test_icdr_two_route_head.py")],
                            cwd=REPO_DIR, env=_env, capture_output=True, text=True)
    _summary = [l for l in _tests.stderr.splitlines() if l.startswith(("Ran ", "OK", "FAILED"))]
    print(f"C1 test suite ({time.time() - _t0:.0f}s):", " | ".join(_summary))
    if _tests.returncode != 0:
        print(_tests.stderr[-6000:])
        raise RuntimeError("The C1 test suite failed on this runtime. Nothing was extracted or fitted.")
else:
    print("Skipped (experiment already completed).")

In [ ]:
# ==== [4] STATUS FROM DISK ====
# Status is derived from disk every time.
#   EXTRACTED    E and its verified parity record are stored -> reused (after re-verification)
#   PARTIAL      an interrupted extraction -> extracted again
#   NOT_STARTED
#   analysis COMPLETED once results.json exists (written last, atomically)

def write_status_table():
    rows = [{"step": f"E extraction seed {s}", "status": exp.embedding_status(EXPERIMENT_DIR, s)}
            for s in exp.BACKBONE_SEEDS]
    rows.append({"step": "fits + scoring + verdict",
                 "status": "COMPLETED" if os.path.exists(posixpath.join(
                     EXPERIMENT_DIR, exp.RESULTS_FILENAME)) else "NOT_STARTED"})
    exp._atomic_json(posixpath.join(EXPERIMENT_DIR, "run_status.json"),
                     {"updated": datetime.datetime.now().isoformat(timespec="seconds"),
                      "steps": rows})
    return rows


for _row in write_status_table():
    print(f"  {_row['step']:<28} {_row['status']}")

In [ ]:
# ==== [5] DATA AND FROZEN REFERENCE ====
# The existing cache archive is only EXTRACTED; no cache is regenerated (a population mismatch stops
# the notebook). H0 = the saved original-head validation outputs; the persistent PDR->=<2 set is
# re-derived from them and must be exactly the 21 images of research record §15.
if not COMPLETED:
    RESULTS = exp.base_results(EXPERIMENT_DIR, REPO_COMMIT, GPUS)
    RESULTS["parameter_parity"] = PARAMETER_PARITY
    DATA = exp.prepare_population(config, itd, msr, SIX_RUN_DIR)
    RESULTS["population"] = DATA["population"]
    print("Population:", json.dumps(DATA["population"]))
    REFERENCE = exp.load_frozen_reference(msr, SIX_RUN_ROOT, DATA["val_ids"], DATA["y_val"])
    RESULTS["persistent_failures"] = {"n": len(REFERENCE["persistent_ids"]),
                                      "image_ids": REFERENCE["persistent_ids"]}
    print(f"H0 loaded for seeds {exp.BACKBONE_SEEDS}; persistent grade-4 -> <=2 set: "
          f"{len(REFERENCE['persistent_ids'])} images")
else:
    print("Skipped (experiment already completed).")

In [ ]:
# ==== [6] FROZEN E EXTRACTION (resumable, one backbone at a time) ====
# Per seed: BEST checkpoint validated against its sealed manifest sha256 -> loaded read-only -> frozen
# (zero trainable variables) -> E extracted for all training and validation images -> parity proven
# (saved logits reproduced; original head on E reproduces the model logits) -> Stage 5-7 + CORN weight
# fingerprints and the weights-file sha256 proven unchanged -> E and its record persisted atomically.
# A seed finished in an earlier session is reused only after its E, population and backbone re-verify.
if not COMPLETED:
    EMBEDDINGS, BACKBONES = {}, {}
    for _seed in exp.BACKBONE_SEEDS:
        _t0 = time.time()
        _e_train, _e_val, BACKBONES[_seed] = exp.extract_backbone_embeddings(
            _seed, EXPERIMENT_DIR, DATA, REFERENCE, SIX_RUN_ROOT)
        EMBEDDINGS[_seed] = (_e_train, _e_val)
        print(f"seed {_seed}: E train {_e_train.shape}, val {_e_val.shape} "
              f"({'resumed' if BACKBONES[_seed]['resumed'] else 'extracted'}, {time.time() - _t0:.0f}s)")
        write_status_table()
    RESULTS["backbones"] = BACKBONES
else:
    print("Skipped (experiment already completed).")

In [ ]:
# ==== [7] FITS, SINGLE VALIDATION SCORING AND THE PRE-REGISTERED VERDICT ====
# Smoke checks on a tiny training subset, then H1 and H2 fitted on TRAINING E only, validation scored
# once, H2 - H1 per seed, paired bootstrap, guardrails, verdict. Outputs are written, with results.json
# LAST (it marks the experiment completed); the full report, including the detailed analysis appendix,
# is printed below.
if not COMPLETED:
    _t0 = time.time()
    RESULTS, TABLE, SAMPLES = exp.run_analysis(
        RESULTS, EMBEDDINGS, DATA["y_train"], DATA["y_val"], REFERENCE["h0_by_seed"],
        REFERENCE["persistent"], DATA["val_ids"], EXPERIMENT_DIR)
    write_status_table()
    print(f"\nAnalysis finished in {time.time() - _t0:.0f}s.")
else:
    print("Skipped (experiment already completed).")

In [ ]:
# ==== [8] FINAL SUMMARY ====
# A completed experiment's stored report is reprinted in full; a run that just finished printed it
# in cell [7], so only the headline is repeated here.
if COMPLETED:
    exp.print_stored_report(EXPERIMENT_DIR)
else:
    _d = RESULTS["decision"]
    print(f"PRE-REGISTERED VERDICT: {_d['verdict']}")
    print("Primary Δ AUROC (H2 − H1), grade 4 vs 0–2, P(grade ≥ 3): " + ", ".join(
        f"seed {s}: {v:+.4f}" for s, v in zip(exp.BACKBONE_SEEDS, _d["per_seed_delta"])))
    print(f"Mean {_d['mean_delta']:+.4f} (SD {_d['sd_delta']:.4f}); paired bootstrap 95% CI "
          f"{_d['bootstrap']['mean_delta_ci'][0]:+.4f} to {_d['bootstrap']['mean_delta_ci'][1]:+.4f}; "
          f"positive in {_d['seeds_positive']}/3 seeds")
    for _k, _g in _d["guardrails"].items():
        print(f"  guardrail {_k:<22} mean Δ {_g['mean_delta']:+.4f}  "
              f"{'holds' if _g['holds'] else 'FAILS'}")
print("\nDDR was not used in this experiment and remains reserved for later external evaluation.")
print("All outputs:", EXPERIMENT_DIR)
for _row in write_status_table():
    print(f"  {_row['step']:<28} {_row['status']}")